In [5]:
import networkx as nx
import torch
from torch_geometric.data import Data

def create_graph_with_structural_features(n):
    # Create a random graph
    G = nx.erdos_renyi_graph(n, p=0.3, seed=42)

    # Node structural features
    degrees = dict(G.degree())
    centrality = nx.betweenness_centrality(G)
    clustering = nx.clustering(G)

    # Find nodes in cycles
    cycles = nx.cycle_basis(G)
    cycle_nodes = set(node for cycle in cycles for node in cycle)

    node_features = []
    labels = []
    for node in G.nodes():
        features = [degrees[node], centrality[node], clustering[node]]
        node_features.append(features)
        labels.append(int(node in cycle_nodes))  # 1 if in cycle, else 0

    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
    if edge_index.numel() == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y)
    return data

# Example usage
data = create_graph_with_structural_features(20)
print(data)
print(data.x)  # Node features
print(data.y)  # Labels

Data(x=[20, 3], edge_index=[2, 67], y=[20])
tensor([[9.0000e+00, 6.1416e-02, 4.1667e-01],
        [4.0000e+00, 7.5977e-03, 3.3333e-01],
        [1.1000e+01, 1.0583e-01, 3.8182e-01],
        [6.0000e+00, 1.5971e-02, 4.6667e-01],
        [5.0000e+00, 1.1598e-02, 4.0000e-01],
        [6.0000e+00, 4.2078e-02, 1.3333e-01],
        [7.0000e+00, 5.4298e-02, 1.9048e-01],
        [4.0000e+00, 1.5432e-02, 3.3333e-01],
        [8.0000e+00, 3.8314e-02, 3.9286e-01],
        [8.0000e+00, 5.1238e-02, 2.8571e-01],
        [6.0000e+00, 2.1759e-02, 4.0000e-01],
        [1.1000e+01, 8.8929e-02, 3.8182e-01],
        [8.0000e+00, 5.8819e-02, 3.2143e-01],
        [6.0000e+00, 1.8730e-02, 5.3333e-01],
        [8.0000e+00, 6.6811e-02, 1.7857e-01],
        [4.0000e+00, 1.9735e-02, 0.0000e+00],
        [6.0000e+00, 2.5135e-02, 2.6667e-01],
        [8.0000e+00, 4.7543e-02, 4.2857e-01],
        [3.0000e+00, 0.0000e+00, 1.0000e+00],
        [6.0000e+00, 3.2391e-02, 4.0000e-01]])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
# =====================================================
# QGNN on Structural Graph Dataset (Single Cell Notebook)
# =====================================================
import os, random, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.nn import Dropout

# -----------------------------------------------------
# Reproducibility
# -----------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------------------------------
# Structural dataset
# -----------------------------------------------------
def create_graph_with_structural_features(n):
    G = nx.erdos_renyi_graph(n, p=0.15, seed=42)
    degrees = dict(G.degree())
    centrality = nx.betweenness_centrality(G)
    clustering = nx.clustering(G)
    cycles = nx.cycle_basis(G)
    cycle_nodes = set(node for cycle in cycles for node in cycle)

    node_features, labels = [], []
    for node in G.nodes():
        features = [degrees[node], centrality[node], clustering[node]]
        node_features.append(features)
        labels.append(int(node in cycle_nodes))

    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
    if edge_index.numel() == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y)
    return data

# -----------------------------------------------------
# QGNN model (unchanged)
# -----------------------------------------------------
class EDUQGCNodeClassifier(nn.Module):
    def __init__(self, n_nodes, in_feats, T=2, seed=0, use_gpu_qnode=True, use_feat_skip=True, num_classes=2):
        super().__init__()
        self.n_nodes = n_nodes
        self.T = T
        self.use_feat_skip = use_feat_skip

        self.enc_W = nn.Parameter(torch.randn(T, 2, in_feats) * 0.08)
        self.enc_b = nn.Parameter(torch.randn(T, 2) * 0.02)

        self.edge_phase  = nn.Parameter(torch.randn(T) * 0.08)
        self.pre_theta   = nn.Parameter(torch.randn(T) * 0.08)
        self.pre_psi     = nn.Parameter(torch.randn(T) * 0.08)
        self.post_theta  = nn.Parameter(torch.randn(T) * 0.08)
        self.post_psi    = nn.Parameter(torch.randn(T) * 0.08)

        readin_dim = 1 + in_feats if use_feat_skip else 1
        self.readout = nn.Sequential(
            nn.Linear(readin_dim, num_classes),
            Dropout(p=0.3)
        )

        use_cuda = torch.cuda.is_available()
        qdev_name = "lightning.gpu" if (use_gpu_qnode and use_cuda) else "default.qubit"
        self.dev = qml.device(qdev_name, wires=n_nodes, shots=None)

        @qml.qnode(self.dev, interface="torch", diff_method="best")
        def circuit(edge_index, X, enc_W, enc_b,
                    edge_phase, pre_theta, pre_psi, post_theta, post_psi):
            for t in range(self.T):
                enc_out = X @ enc_W[t].T + enc_b[t]
                alphas = enc_out[:, 0]; betas = enc_out[:, 1]
                for i in range(self.n_nodes):
                    qml.RX(alphas[i], wires=i)
                    qml.RY(betas[i], wires=i)

                for i in range(self.n_nodes):
                    qml.RZ(pre_psi[t], wires=i)
                    qml.RX(pre_theta[t], wires=i)

                E = edge_index.shape[1]
                for e in range(E):
                    u = int(edge_index[0, e].item()); v = int(edge_index[1, e].item())
                    if u != v:
                        qml.ControlledPhaseShift(edge_phase[t], wires=[u, v])

                for i in range(self.n_nodes):
                    qml.RZ(post_psi[t], wires=i)
                    qml.RX(post_theta[t], wires=i)

            return [qml.expval(qml.Z(i)) for i in range(self.n_nodes)]

        self._circuit = circuit

    def forward(self, edge_index_torch, x_torch):
        device = next(self.parameters()).device
        edge_index = edge_index_torch.to(device)
        X = x_torch.to(device).float()

        out = self._circuit(edge_index, X,
                            self.enc_W, self.enc_b,
                            self.edge_phase,
                            self.pre_theta, self.pre_psi,
                            self.post_theta, self.post_psi)

        expvals = torch.stack(out, dim=0).float().to(device)
        if expvals.dim() == 1:
            expvals = expvals.unsqueeze(1)
        elif expvals.dim() == 2 and expvals.shape[1] == 1:
            pass
        else:
            expvals = expvals.squeeze(-1).unsqueeze(1)

        readin = torch.cat([expvals, X], dim=1) if self.use_feat_skip else expvals
        logits = self.readout(readin)
        return logits

# -----------------------------------------------------
# Masks (stratified train/val/test split)
# -----------------------------------------------------
def stratified_masks(y, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, seed=42):
    idx = np.arange(len(y))
    idx_train, idx_tmp, y_train, y_tmp = train_test_split(
        idx, y, stratify=y, train_size=train_ratio, random_state=seed
    )
    idx_val, idx_test, _, _ = train_test_split(
        idx_tmp, y_tmp, stratify=y_tmp, train_size=val_ratio/(val_ratio+test_ratio), random_state=seed
    )
    train_mask = torch.zeros(len(y), dtype=torch.bool)
    val_mask   = torch.zeros(len(y), dtype=torch.bool)
    test_mask  = torch.zeros(len(y), dtype=torch.bool)
    train_mask[idx_train] = True; val_mask[idx_val] = True; test_mask[idx_test] = True
    return train_mask, val_mask, test_mask

# -----------------------------------------------------
# Evaluation helper
# -----------------------------------------------------
@torch.no_grad()
def evaluate(model, data, mask):
    model.eval()
    logits = model(data.edge_index, data.x)
    loss = F.cross_entropy(logits[mask], data.y[mask])
    preds = logits.argmax(dim=1)
    acc = (preds[mask] == data.y[mask]).float().mean().item()
    return loss.item(), acc, preds.cpu().numpy()

# -----------------------------------------------------
# Experiment
# -----------------------------------------------------
def run_structural_experiment(n=20, epochs=5, lr=0.001, seed=42):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    def create_balanced_graph(x, p, min_ratio=0.3, max_ratio=0.7):
        while True:
            data = create_graph_with_structural_features(x, p)
            ratio = data.y.sum().item() / len(data.y)
            if min_ratio <= ratio <= max_ratio:
                return data

    data = create_balanced_graph(x=20, p=0.15)
    data.edge_index = to_undirected(data.edge_index)
    data.train_mask, data.val_mask, data.test_mask = stratified_masks(data.y.numpy(), seed=seed)
    data = data.to(device)

    model = EDUQGCNodeClassifier(
        n_nodes=data.num_nodes,
        in_feats=data.num_features,
        num_classes=2
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    print(f"Training QGNN on structural dataset with {data.num_nodes} nodes.")
    # Print dataset details
    print(f"Dataset details:")
    print(f" - Number of nodes: {data.num_nodes}")
    print(f" - Number of edges: {data.edge_index.shape[1]}")
    print(f" - Number of features: {data.num_features}")
    print(f" - Number of classes: {len(torch.unique(data.y)) if hasattr(data, 'y') else 'unknown'}")
    print(data)
    print(data.x)
    print(data.y)
    for epoch in range(1, epochs+1):
        model.train()
        logits = model(data.edge_index, data.x)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        opt.zero_grad(); loss.backward(); opt.step()

        val_loss, val_acc, _ = evaluate(model, data, data.val_mask)
        print(f"Epoch {epoch:03d} | tr_loss={loss.item():.3f} | val_loss={val_loss:.3f} | val_acc={val_acc:.3f}")

    test_loss, test_acc, preds = evaluate(model, data, data.test_mask)
    print(f"\n[Test] loss={test_loss:.3f} acc={test_acc:.3f}")
    y_true = data.y[data.test_mask].cpu().numpy()
    print(classification_report(y_true, preds[data.test_mask.cpu().numpy()], zero_division=0))

    return model, data

# =====================================================
# Run Example
# =====================================================
model, data = run_structural_experiment(n=20, epochs=50
                                        , seed=42)


TypeError: create_graph_with_structural_features() takes 1 positional argument but 2 were given

: 

In [7]:
unique, counts = torch.unique(data.y, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))


{1: 20}
